Complete pipeline for RAG

Data_Injection

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Pazhani vel B\AppData\Local\Temp\ipykernel_17576\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\Pazhani vel B\OneDrive\Desktop\AI_ML_Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### reading all the information from the all pdf

def process_all_pdf(pdf_dirc):
    """Process all the pdf"""

    all_document = []
    pdf_dir = Path(pdf_dirc)

    ### finding all the path recusrsively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            ## adding the new information to metadata

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_document.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error : {e}")

    
    print(f"\nTotal documents loaded: {len(all_document)}")

    return all_document



## processing all the pdf in that dir

all_pdf_doc = process_all_pdf("../Data/pdffile")
print(all_pdf_doc)


Found 5 PDF files to process

Processing : BIO-DATA-AND-SOP-YNU-2026.pdf
 Loaded 4 pages

Processing : Dept_info.pdf
 Loaded 8 pages

Processing : DocScanner 10-Jun-2026 4-36 pm.pdf
 Loaded 1 pages

Processing : Module 1 - Introduction to Amazon Web Services.pdf
 Loaded 4 pages

Processing : stydynotes.pdf
 Loaded 110 pages

Total documents loaded: 127
[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-16T18:16:53+00:00', 'author': 'Pazhani Vel.B', 'moddate': '2026-06-16T18:16:53+00:00', 'source': '..\\Data\\pdffile\\BIO-DATA-AND-SOP-YNU-2026.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'BIO-DATA-AND-SOP-YNU-2026.pdf', 'file_type': 'pdf'}, page_content='BIO-DATA \nYNU International Symposium 2026 (Yokohama -SXIP Program) \n1. Personal Details \nName: B. Pazhani Vel \nContact Number: +91 9363905431 \nEmail Address: pazhanivel680@gmail.com  \nGitHub: github.com/pazhani-vel \nLinkedIn: linkedin/pazhanivel

Chuncking

In [3]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Splitting the data into chunck"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} document into {len(split_docs)} chunks")

    # showign example

    if split_docs:
        print(f"\nExample chunk")
        print(f"content : {split_docs[0].page_content[:200]}....")
        print(f"MetaData : {split_docs[0].metadata}")

    return split_docs

In [4]:
chunks = split_documents(all_pdf_doc)
print(chunks)

Split 127 document into 26 chunks

Example chunk
content : BIO-DATA 
YNU International Symposium 2026 (Yokohama -SXIP Program) 
1. Personal Details 
Name: B. Pazhani Vel 
Contact Number: +91 9363905431 
Email Address: pazhanivel680@gmail.com  
GitHub: github.....
MetaData : {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-16T18:16:53+00:00', 'author': 'Pazhani Vel.B', 'moddate': '2026-06-16T18:16:53+00:00', 'source': '..\\Data\\pdffile\\BIO-DATA-AND-SOP-YNU-2026.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'BIO-DATA-AND-SOP-YNU-2026.pdf', 'file_type': 'pdf'}
[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-16T18:16:53+00:00', 'author': 'Pazhani Vel.B', 'moddate': '2026-06-16T18:16:53+00:00', 'source': '..\\Data\\pdffile\\BIO-DATA-AND-SOP-YNU-2026.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'BIO-DATA-AND-SOP-YNU-2026.pdf', 'file_type

Embedding and vectorDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer       ## hugging face model
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics. pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):   ## constructor

        """Initialize the embedding manager
        Args:
        model_name: Hugging Face model name for sentence embeddings"""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            print (f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print (f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
        texts: List of text strings to embed
        Returns:
        numpy array of embeddings with shape (len(texts), embedding_dim) """
        if not self.model:
            raise ValueError("Model not loaded")
        print (f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print (f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

## initializing the embedding model

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2083.46it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Pazhani vel B\AppData\Local\Temp\ipykernel_17576\2195602465.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print (f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Vector DB

In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../Data/vector_store"):
        """Initialize the vector store I
        Args:
        collection_name: Name of the ChromaDB collection
        persist_directory: Directory to persist the vector store"""
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

            # Create persistent ChromaDB client

        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        """Add documents and their embeddings to the vector store

        Args:
        documents: List of LangChain documents
        embeddings: Corresponding embeddings for the documents"""

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store ... ")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id =f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            # Add to collection

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


Vectorstore = VectorStore()
Vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 26


In [8]:
### created chunks

chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-16T18:16:53+00:00', 'author': 'Pazhani Vel.B', 'moddate': '2026-06-16T18:16:53+00:00', 'source': '..\\Data\\pdffile\\BIO-DATA-AND-SOP-YNU-2026.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'BIO-DATA-AND-SOP-YNU-2026.pdf', 'file_type': 'pdf'}, page_content='BIO-DATA \nYNU International Symposium 2026 (Yokohama -SXIP Program) \n1. Personal Details \nName: B. Pazhani Vel \nContact Number: +91 9363905431 \nEmail Address: pazhanivel680@gmail.com  \nGitHub: github.com/pazhani-vel \nLinkedIn: linkedin/pazhanivel  \nRoll No : 2024506066 \nDate of Birth: 01/06/2007 \nGender: Male \nNationality: Indian \nPassport Status: Not Available \n \n2. Academic Details \nInstitution: Madras Institute of Technology (MIT Campus), Anna University  \nProgramme: Bachelor of Technology – Information Technology  \nYear of Study: 3 Year (2024–2028 Batch) \nCurrent CGPA: 9.7 / 10 \

In [9]:
### converting the text to embedding
texts = [doc.page_content for doc in chunks]


### generating the embedding
embedding = embedding_manager.generate_embeddings(texts)

### storing in the vector store
vectorDB = Vectorstore.add_documents(chunks,embedding)

Generating embeddings for 26 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.59s/it]

Generated embeddings with shape: (26, 384)
Adding 26 documents to vector store ... 
Successfully added 26 documents to vector store
Total documents in collection: 52


Retrival pipeline on RAG

In [10]:
class RAGRetriever:
    """Handles query-basad retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriever

        Args:
        vector_store: Vector store containing document embeddings
        embedding_manager: Manager for generating query embeddings"""

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        """Retrieve relevant documents for a query

        Args:
        query: The search query
        top_k: Number of top results to return
        score_threshold: Minimum similarity score threshold

        Returns:
        List of dictionaries containing retrieved documents and metadata"""

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                        'id': doc_id,
                        'content': document,
                        'metadata': metadata,
                        'similarity_score': similarity_score,
                        'distance': distance,
                        'rank': i + 1})

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrival {e}")
            return []
        


In [11]:
retrieveObj = RAGRetriever(Vectorstore,embedding_manager)
retrieveObj

In [12]:
retrieveObj.retrieve("what is dsa")

Retrieving documents for query: 'what is dsa'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.04it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_1bb014f8_24',
  'content': 'DSADSA\nComplete Notes on',
  'metadata': {'source': '..\\Data\\pdffile\\stydynotes.pdf',
   'producer': 'Pdftools SDK 1.5.1.0, Pdftools (www.pdf-tools.com)',
   'source_file': 'stydynotes.pdf',
   'page_label': '1',
   'page': 0,
   'creator': 'PyPDF',
   'content_length': 24,
   'total_pages': 110,
   'moddate': '2024-07-02T04:13:02+00:00',
   'doc_index': 24,
   'file_type': 'pdf',
   'creationdate': ''},
  'similarity_score': 0.1840357780456543,
  'distance': 0.8159642219543457,
  'rank': 1},
 {'id': 'doc_7d5c42a0_24',
  'content': 'DSADSA\nComplete Notes on',
  'metadata': {'content_length': 24,
   'total_pages': 110,
   'page': 0,
   'doc_index': 24,
   'file_type': 'pdf',
   'creator': 'PyPDF',
   'page_label': '1',
   'producer': 'Pdftools SDK 1.5.1.0, Pdftools (www.pdf-tools.com)',
   'creationdate': '',
   'source': '..\\Data\\pdffile\\stydynotes.pdf',
   'source_file': 'stydynotes.pdf',
   'moddate': '2024-07-02T04:13:02+00:00'},
  's

In [16]:
## complete pipeline for intergrating the retrival to llm output

### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    print("\nContext:")
    print(context)
    
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely. Give the confidence level of the answer with the accuracy level out of 100.
        Context:
        {context}
        Question: {query}
        Answer:"""

    response=llm.invoke([prompt. format(context=context, query=query)])
    return response.content

    I

In [18]:
answer = rag_simple("Benifit of cloud computing",retrieveObj,llm)
print("\nResponse:")
print(answer)

Retrieving documents for query: 'Benifit of cloud computing'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.20it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

Context:
Increase speed and agility 
The flexibility of cloud computing makes it easier for you to develop and deploy 
applications. This flexibility provides you with more time to experiment and innovate. When 
computing in data centres, it may take weeks to obtain new resources that you need. By 
comparison, cloud computing enables you to access new resources within minutes. 
 
Go global in minutes 
The global footprint of the AWS Cloud enables you to deploy applications to customers 
around the world quickly, while providing them with low latency. This means that even if 
you are located in a different part of the world than your customers, customers are able to 
access your applications with minimal delays. Later in this course, you will explore the AWS 
global infrastructure in greater detail. You will examine some of the services that you can 
use to deliver content to customers around the world. 



Response:
The benefits of cloud computing include: 
1. Increased speed and agility 
2. Variable expense, saving on costs 
3. Ability to focus less on infrastructure management and more on applications and customers 
4. No need to predict infrastructure capacity before deploying an application 
5. Pay only for the resources used.

Confidence level: 95 
Accuracy level: 95/100
